In [3]:
# Libraries

import time
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pyglam

In [4]:
# Config

CSV_PATH = r"C:\Users\Renata Pensin\Downloads\training_pce_dataset.csv"
FEATURES = ["fck_mpa", "rh_percent", "cover_mm", "tempo"]
TARGETS = ["lambda_1", "lambda_2"]

TRAIN_SIZE_TOTAL = 25000   # subamostra total (treino+teste) tirada do dataset original
TEST_FRAC = 0.2            # 20% dessa subamostra vira teste
RANDOM_STATE = 42

# Carregar os dados e preparar para treino/teste

print("Carregando e preparando os dados...")
df = pd.read_csv(CSV_PATH)

X = df[FEATURES].values
y = df[TARGETS].values

# Subamostragem 
X_sub, _, y_sub, _ = train_test_split(
    X, y, train_size=TRAIN_SIZE_TOTAL, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=TEST_FRAC, random_state=RANDOM_STATE
)

# Padronização 
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Tamanho do Treino: {X_train.shape[0]} amostras")
print(f"Tamanho do Teste: {X_test.shape[0]} amostras\n")

def evaluate_single_target(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    print(f"[{name}] R2: {r2:.6f} | MAE: {mae:.5f} | RMSE: {rmse:.5f}")

Carregando e preparando os dados...
Tamanho do Treino: 20000 amostras
Tamanho do Teste: 5000 amostras



In [5]:
# Treinando Rede Neural para lambda_1

print("Treinando Rede Neural para lambda_1...")
t0 = time.time()

mlp_l1 = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=RANDOM_STATE,
)
# Treinando apenas com a primeira coluna do y (lambda_1)
mlp_l1.fit(X_train_s, y_train[:, 0])
pred_l1 = mlp_l1.predict(X_test_s)

evaluate_single_target("Lambda 1", y_test[:, 0], pred_l1)
print(f"Tempo: {time.time() - t0:.1f}s, Iterações: {mlp_l1.n_iter_}\n")

Treinando Rede Neural para lambda_1...
[Lambda 1] R2: 0.999495 | MAE: 0.30555 | RMSE: 0.40569
Tempo: 8.9s, Iterações: 57



In [6]:
# Treinando Rede Neural para lambda_2

print("Treinando Rede Neural para lambda_2...")
t0 = time.time()

mlp_l2 = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=RANDOM_STATE,
)
# Treinando apenas com a segunda coluna do y (lambda_2)
mlp_l2.fit(X_train_s, y_train[:, 1])
pred_l2 = mlp_l2.predict(X_test_s)

evaluate_single_target("Lambda 2", y_test[:, 1], pred_l2)
print(f"Tempo: {time.time() - t0:.1f}s, Iterações: {mlp_l2.n_iter_}\n")

Treinando Rede Neural para lambda_2...
[Lambda 2] R2: 0.999196 | MAE: 0.01312 | RMSE: 0.01777
Tempo: 8.3s, Iterações: 63



In [7]:
# Salvar os modelos 

joblib.dump(mlp_l1, "model_mlp_lambda1.pkl")
joblib.dump(mlp_l2, "model_mlp_lambda2.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Treinamento concluído!")
print("Arquivos salvos:")
print(" - model_mlp_lambda1.pkl")
print(" - model_mlp_lambda2.pkl")
print(" - scaler.pkl")

Treinamento concluído!
Arquivos salvos:
 - model_mlp_lambda1.pkl
 - model_mlp_lambda2.pkl
 - scaler.pkl


In [8]:
# Carregar os modelos 
try:
    mlp_l1 = joblib.load("model_mlp_lambda1.pkl")
    mlp_l2 = joblib.load("model_mlp_lambda2.pkl")
    scaler = joblib.load("scaler.pkl")
    print("Modelos carregados com sucesso!")
except FileNotFoundError:
    print("Erro: Arquivos .pkl não encontrados. Rode o treinamento primeiro.")

Modelos carregados com sucesso!


In [9]:


# Definir as condições fixas para a predição

fck_input = 30.0
rh_input = 65.0
cover_input = 40.0

# time list
lista_tempos = [0.0, 16.666666666666668, 33.333333333333336, 50.0, 66.66666666666667, 83.33333333333334, 100.0, 116.66666666666667, 133.33333333333334, 150.0]

# Cria um DataFrame repetindo as condições fixas para cada tempo da lista
novos_dados = pd.DataFrame({
    "fck_mpa": [fck_input] * len(lista_tempos),
    "rh_percent": [rh_input] * len(lista_tempos),
    "cover_mm": [cover_input] * len(lista_tempos),
    "tempo": lista_tempos
})

print(f"\nRealizando predições para Fck={fck_input}MPa, RH={rh_input}%, Cobrimento={cover_input}mm...\n")


Realizando predições para Fck=30.0MPa, RH=65.0%, Cobrimento=40.0mm...



In [10]:
# Escalonar os novos dados e fazer as predições

X_novos_escalonados = scaler.transform(novos_dados)

# Fazemos as predições separadas
novos_dados["pred_lambda_1"] = mlp_l1.predict(X_novos_escalonados)
novos_dados["pred_lambda_2"] = mlp_l2.predict(X_novos_escalonados)

# O dataframe agora devolve a relação completa do tempo vs lambdas
print(novos_dados.to_string(index=False))

 fck_mpa  rh_percent  cover_mm      tempo  pred_lambda_1  pred_lambda_2
    30.0        65.0      40.0   0.000000      39.599110       1.798032
    30.0        65.0      40.0  16.666667      27.423679       1.626900
    30.0        65.0      40.0  33.333333      22.786891       1.450434
    30.0        65.0      40.0  50.000000      18.330571       1.320767
    30.0        65.0      40.0  66.666667      14.943080       1.237512
    30.0        65.0      40.0  83.333333      12.195621       1.161478
    30.0        65.0      40.0 100.000000       9.524363       1.080813
    30.0        65.0      40.0 116.666667       6.974053       1.002097
    30.0        65.0      40.0 133.333333       4.467524       0.931302
    30.0        65.0      40.0 150.000000       2.190534       0.869596


c:\Users\Renata Pensin\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
